# Simplified Lead Scoring Model 

## Design Philosophy

This model is a distilled version of a comprehensive lead scoring system, reduced to 5 core parameters that capture what actually matters for  closing deals:

| # | Parameter | What it answers |
|---|-----------|-----------------|
| P1 | Money | Can they pay? |
| P2 | Fit | Does our product solve their problem? |
| P3 | Power | Are we talking to the decision-maker? |
| P4 | Momentum | Is this deal moving forward? |
| P5 | Edge | Why would they choose us? |

---

## Key Design Decisions

### Unbounded Score (No ceiling of 100)

Each parameter produces a log-point value using the natural logarithm:

$$L_i = \ln(1 + R_i)$$

where $$R_i$$ is a raw score that can grow without limit. The logarithm ensures:

- The score has no upper bound — it keeps climbing as the lead improves.
- There are diminishing returns — doubling a metric doesn't double the score, but it does increase it meaningfully.
- The score stays interpretable — it doesn't explode to thousands.

### Final Score Formula

$$S = G \times M \times \sum_{i=1}^{5} w_i \cdot L_i$$

Where:

- $G$ = gate (binary: 0 if any knockout condition is triggered, 1 otherwise)
- $M$ = deal friction modifier $\in [0.5, 1.0]$
- $w_i$ = parameter weight
- $L_i$ = log-point score for parameter $$i$$



## Setup: Import Libraries and Define Helper Functions

We use three important mathematical building blocks:

1. **BCI (Bayesian Credible Interval)** — When we have historical data (e.g., "8 out of 12 similar deals had confirmed budgets"), BCI gives us a conservative lower-bound estimate rather than a naive ratio. This prevents overconfidence from small samples.

2. **EDP (Exponential Decay Pressure)** — Models urgency and competitive pressure. The formula $$e^{-k \cdot x}$$ produces values that drop steeply at first, then flatten. Used for timeline pressure (more days = less urgency) and competitive pressure (more competitors = worse position).

3. **Log-Norm** — Normalizes counts (like number of stakeholders) with diminishing returns. The 5th stakeholder adds less value than the 2nd.


In [70]:
import numpy as np
from scipy.stats import beta as beta_dist

def bci(successes: int, trials: int, confidence: float = 0.05) -> float:
    if trials <= 0:
        return 0.0
    return beta_dist.ppf(confidence, successes + 1, trials - successes + 1)


def edp(x: float, k: float) -> float:
    return np.exp(-k * x)


def lognorm(count: float, cap: float) -> float:
    if cap <= 0:
        return 0.0
    return min(np.log(1 + count) / np.log(1 + cap), 1.0)


In [71]:
params = {
    # ── P1: Money  ──
    "budget": 50000,                # Total budget the buyer has allocated or can realistically spend for this purchase.
                                     # Increasing this improves affordability and deal safety; decreasing it raises pricing risk and budget objections.

    "deal_value": 45000,            # Expected contract value or selling price of this specific deal.
                                     # Increasing this raises potential revenue but can hurt close probability if it nears/exceeds budget; decreasing it may improve affordability but lowers upside.

    "budget_confirmation": 0.75,    # Confidence that the budget is explicitly confirmed, approved, and usable for this deal.
                                     # Increasing this reduces uncertainty and strengthens forecast confidence; decreasing it means more risk that "budget exists" only verbally or not at all.

    "funding_security": 0.7,        # Stability of the funding source behind the deal, such as approved departmental or project-backed money.
                                     # Increasing this makes the deal more dependable; decreasing it suggests the money could be delayed, reallocated, or withdrawn.

    "n_confirmed_budget": 8,        # Number of past opportunities with clearly confirmed budget used as evidence/calibration data.
                                     # Increasing this improves confidence in how this factor is modeled; decreasing it means weaker historical support for budget-related assumptions.

    "n_similar_deals": 12,          # Number of comparable historical deals used to benchmark pricing, budget fit, and close behavior.
                                     # Increasing this makes your budget/value expectations more reliable; decreasing it makes estimates noisier and less defensible.

    # ── P2: Fit  ──
    "solution_fit": 0.9,            # How well your product solves the customer's actual use case, needs, and desired outcomes.
                                     # Increasing this boosts product relevance and win likelihood; decreasing it creates mismatch, objections, and higher competitive risk.

    "problem_clarity": 0.7,         # How clearly the customer understands, defines, and agrees on the problem they need to solve.
                                     # Increasing this sharpens urgency and buying intent; decreasing it leads to confusion, slower cycles, and weaker prioritization.

    "n_features_available": 17,     # Number of required or valued customer features your product already supports.
                                     # Increasing this strengthens perceived coverage and readiness; decreasing it exposes product gaps and can trigger no-decision or competitive loss.

    "n_features_needed": 20,        # Total number of important features the customer expects for a workable solution.
                                     # Increasing this raises the bar you must meet and can make the sale harder; decreasing it simplifies the fit requirement and usually improves closeability.

    "dissatisfaction": 0.6,         # Degree of frustration with the customer's current solution, process, vendor, or status quo.
                                     # Increasing this creates more urgency to change; decreasing it makes the status quo easier to tolerate and harder to displace.

    "icp_match": 0.85,              # How closely this account matches your ideal customer profile in size, industry, maturity, and use case.
                                     # Increasing this usually improves win rate, retention, and value realization; decreasing it often means more friction and lower predictability.

    "roi_quantified": 0.7,          # How well the financial or business return from your solution has been calculated and documented with the buyer.
                                     # Increasing this strengthens the business case and executive support; decreasing it makes value feel vague and easier to deprioritize.

    "metric_agreed": 0.6,           # Degree to which both you and the customer agree on the success metrics that define value.
                                     # Increasing this aligns expectations and helps justify the purchase; decreasing it causes ambiguity around outcomes and weakens the case for action.

    "cost_of_inaction": 0.5,        # How clearly the downside of doing nothing has been identified, quantified, and accepted by the buyer.
                                     # Increasing this increases urgency and pushes decisions forward; decreasing it makes delay feel safe and reduces pressure to buy now.

    # ── P3: Power  ──
    "seniority": 0.8,               # Relative organizational seniority of your main contact or sponsor.
                                     # Increasing this usually means more influence and faster escalation; decreasing it often means less authority and more internal dependency.

    "decision_involvement": 1.0,    # How directly the contact participates in evaluation, recommendation, or final approval.
                                     # Increasing this improves access to real decision dynamics; decreasing it raises the chance you're relying on a peripheral contact.

    "n_stakeholders": 4,            # Number of people or teams meaningfully involved in approving or influencing the deal.
                                     # Increasing this can improve coverage if managed well but usually adds complexity and slows consensus; decreasing it simplifies coordination but may hide unseen blockers.

    "org_alignment": 0.8,           # Level of internal agreement across stakeholders that the problem, solution, and priority are legitimate.
                                     # Increasing this reduces internal friction and improves execution; decreasing it creates mixed signals, delays, and political resistance.

    "access_to_power": 0.7,         # Quality of your direct or indirect access to the real decision-makers and budget holders.
                                     # Increasing this improves control, discovery quality, and negotiation clarity; decreasing it leaves you dependent on secondhand information.

    "champion_identified": 1,       # Whether a true internal champion has been clearly identified within the customer organization.
                                     # Increasing from 0 to 1 is a major positive because it means someone can sell internally for you; decreasing to 0 means you may lack internal momentum.

    "champion_strength": 0.75,      # How credible, influential, and persuasive that champion is inside the account.
                                     # Increasing this improves your internal reach and deal resilience; decreasing it weakens advocacy and makes resistance harder to overcome.

    "champion_active": 0.8,         # How actively the champion is taking actions such as aligning stakeholders, sharing info, and driving next steps.
                                     # Increasing this accelerates deal progress; decreasing it suggests passive support, stalled motion, or a fragile champion.

    "econ_buyer_identified": 1,     # Whether the economic buyer, the person who can truly approve spending, is known.
                                     # Increasing from 0 to 1 reduces guesswork around approval; decreasing to 0 means you may be building a case without the real buyer in view.

    "econ_buyer_engaged": 0.65,     # Degree to which the economic buyer is directly engaged, informed, and responsive in the process.
                                     # Increasing this improves approval odds and forecast quality; decreasing it raises the risk of late-stage rejection or silent stalling.

    # ── P4: Momentum ──
    "days_to_decision": 45,         # Expected number of days until the customer is likely to make a decision.
                                     # Increasing this usually signals a slower sales cycle and more delay risk; decreasing it generally indicates stronger urgency and faster movement.

    "engagement_score": 7.0,        # Composite score reflecting meeting quality, responsiveness, activity level, and overall buying engagement.
                                     # Increasing this suggests stronger interest and momentum; decreasing it can indicate fading attention, weaker intent, or impending stall.

    "velocity": 1.5,                # Speed at which the deal is progressing through meetings, milestones, and decision stages.
                                     # Increasing this shows faster forward motion and healthier momentum; decreasing it points to friction, indecision, or process slowdown.

    "next_step_defined": 1,         # Whether there is a clear, mutually agreed next action with ownership and timing.
                                     # Increasing from 0 to 1 improves control and reduces drift; decreasing to 0 often means the deal may stall between interactions.

    "trigger_event": 1,             # Whether a meaningful business event exists that is forcing or motivating a decision, such as growth, renewal, or compliance pressure.
                                     # Increasing from 0 to 1 strengthens urgency; decreasing to 0 removes external pressure and makes postponement more likely.

    "decision_process_mapped": 0.7, # How well you understand the customer's actual decision path, stakeholders, approval flow, and evaluation steps.
                                     # Increasing this improves predictability and execution; decreasing it raises the risk of surprises, missed steps, and forecast error.

    "procurement_stage": 0.4,       # Progress of the deal through procurement, legal, vendor onboarding, or purchasing workflows.
                                     # Increasing this means the deal is advancing operationally toward close; decreasing it indicates earlier-stage process risk and more remaining friction.

    "paper_process_clear": 0.5,     # Clarity around paperwork requirements such as legal review, security review, MSA, PO, and contracting steps.
                                     # Increasing this reduces closing friction and timeline surprises; decreasing it means more uncertainty and a higher chance of end-stage delays.

    # ── P5: Edge ──
    "win_rate": 0.6,                # Historical win percentage in similar competitive situations or account types.
                                     # Increasing this suggests stronger repeatability and a better baseline chance to win; decreasing it implies weaker positioning or execution in similar deals.

    "differentiation": 0.8,         # How clearly your solution stands apart from alternatives in ways the customer values.
                                     # Increasing this makes it easier to justify selection and pricing; decreasing it pushes the deal toward price comparison and commoditization.

    "relationship_strength": 0.7,   # Overall trust, rapport, and quality of working relationships with key people in the account.
                                     # Increasing this improves access, honesty, and resilience under pressure; decreasing it makes the deal more vulnerable to competitors and internal doubt.

    "n_competitors": 3,             # Number of credible competing vendors or alternatives being considered.
                                     # Increasing this usually lowers win probability and increases comparison pressure; decreasing it improves focus and reduces competitive noise.

    "sole_vendor": 0,               # Whether you are the only vendor being considered in the process.
                                     # Increasing from 0 to 1 is strongly positive because it removes direct competition; decreasing to 0 means the deal becomes contested again.

    "n_competitive_deals": 30,      # Number of historical competitive deals used to estimate win patterns and competitive behavior.
                                     # Increasing this improves confidence in competitive benchmarks; decreasing it makes win-rate and edge assumptions less statistically reliable.

    "n_wins": 18,                   # Number of wins within the comparable historical competitive deal sample.
                                     # Increasing this improves your observed evidence of success; decreasing it lowers the empirical support behind your win-rate expectations.
}


gates = [True, True, True] 
modifier = 0.9          

## Parameter 1: Money 

**Question answered:** Can this prospect realistically pay, and how much room is there?

### Variables

| Symbol | Description | Source |
|--------|-------------|--------|
| $$B$$ | Estimated budget available | Sales input |
| $$D$$ | Deal value (our asking price) | CRM |
| $$C_{stated}$$ | Budget confirmation level $$[0, 1]$$ | Sales input |
| $$F$$ | Funding security $$[0, 1]$$ | Sales input |

### How It Works

1. **Budget Ratio** $$B_r = B / D$$ — Intentionally unbounded. If the prospect has 3x your deal value in budget, that's a genuinely stronger signal than exactly 1x.

2. **Confirmation Adjustment** — Blends the rep's stated confidence with Bayesian evidence from historical data (if available).

$C_{adj} = 0.5 \cdot C_{stated} + 0.5 \cdot C_{evidence}$

3. **Raw Score** — Budget ratio amplified by how confirmed and secure the money is:

$$R_1 = B_r \cdot (0.6 \cdot C_{adj} + 0.4 \cdot F)$$

4. **Log-Point** — $$L_1 = \ln(1 + R_1)$$

In [72]:
def compute_p1_money(B: float, D: float, C_stated: float, F: float,
                     n_confirmed: int = None, n_similar: int = None):
    Br = B / D

    if n_confirmed is not None and n_similar is not None and n_similar > 0:
        C_evidence = bci(n_confirmed, n_similar)
        C_adj = 0.5 * C_stated + 0.5 * C_evidence
    else:
        C_adj = C_stated 

    R1 = Br * (0.6 * C_adj + 0.4 * F)

    L1 = np.log(1 + R1)

    details = {
        "budget_ratio": Br,
        "C_adj": C_adj,
        "R1_raw": R1,
        "L1_score": L1,
    }
    return L1

# Parameter 2: Fit

**Question answered:** Does our product solve their problem, and are they the right kind of customer?

## Variables

| Symbol     | Description                                               | Source              |
| ---------- | --------------------------------------------------------- | ------------------- |
| $S$        | Solution fit \[0, 1]                                      | Sales/SE assessment |
| $P$        | Problem clarity \[0, 1]                                   | Sales input         |
| $T\_{adj}$ | Technical coverage = features available / features needed | Product team        |
| $D$        | Dissatisfaction with current solution \[0, 1]             | Sales input         |
| $I$        | ICP match \[0, 1]                                         | Marketing/Ops       |
| $ROI$      | ROI quantified \[0, 1]                                    | Sales/SE assessment |
| $MA$       | Metric agreed \[0, 1]                                     | Sales input         |
| $COI$      | Cost of inaction \[0, 1]                                  | Sales input         |

## How It Works

**1. Technical Coverage:**

$$T_{adj} = n_{available} / n_{needed}  $$

Unbounded. (25 features for a 20-feature need is 1.25).

**2. Metrics Composite:**

$$Metrics = 0.40 \cdot ROI + 0.35 \cdot MA + 0.25 \cdot COI$$

**3. Core Fit Score:**

$$F_{core} = 0.25 \cdot S + 0.20 \cdot P + 0.20 \cdot T_{adj} + 0.15 \cdot D + 0.20 \cdot Metrics$$

**4. ICP Amplifier — Acts as a multiplier (up to +50%), never a zeroing factor:**

$$R_2 = F_{core} \cdot (1 + 0.5 \cdot I)$$

**5. Log-Point — $L_2 = \ln(1 + R_2)$**



In [73]:
def compute_p2_fit(S: float, P: float, n_feat_available: int,
                   n_feat_needed: int, D: float, I: float,
                   roi_quantified: float, metric_agreed: float,
                   cost_of_inaction: float):

    Tadj = n_feat_available / max(n_feat_needed, 1)

    Metrics = (0.40 * roi_quantified
               + 0.35 * metric_agreed
               + 0.25 * cost_of_inaction)

    Fcore = (0.25 * S
             + 0.20 * P
             + 0.20 * Tadj
             + 0.15 * D
             + 0.20 * Metrics)

    R2 = Fcore * (1 + 0.5 * I)
    L2 = np.log(1 + R2)

    details = {
        "Tadj": Tadj,
        "Metrics": Metrics,
        "Fcore": Fcore,
        "R2_raw": R2,
        "L2_score": L2,
    }
    return L2

# Parameter 3: Power

**Question answered:** Are we talking to someone who can actually make this happen?

This carries the highest weight (tied with Money) because we can't afford long sales cycles to the wrong person.

## Variables

| Symbol      | Description                                  | Source      |
| ----------- | -------------------------------------------- | ----------- |
| $A$         | Authority = seniority x decision involvement | Sales input |
| $N$         | Number of stakeholders identified            | Sales input |
| $O$         | Organizational alignment \[0, 1]             | Sales input |
| $Ac$        | Direct access to decision-maker \[0, 1]      | Sales input |
| $Ch\_{id}$  | Champion identified {0, 1}                   | Sales input |
| $Ch\_{str}$ | Champion strength \[0, 1]                    | Sales input |
| $Ch\_{act}$ | Champion active \[0, 1]                      | Sales input |
| $EB\_{id}$  | Economic buyer identified {0, 1}             | Sales input |
| $EB\_{eng}$ | Economic buyer engaged \[0, 1]               | Sales input |


## How It Works

1. **Authority** $A = seniority \times involvement$ — the base authority signal.

2. **Stakeholder Breadth** — log-normalized, capped at 5:

$$N_{score} = \frac{\ln(1+N)}{\ln(1+5)}$$

3. **Champion Score:**

$$Champ = Ch_{id} \cdot (0.50 \cdot Ch_{str} + 0.50 \cdot Ch_{act})$$

4. **Economic Buyer Score:**

$$EB = EB_{id} \cdot EB_{eng}$$

5. **Authority Blend** — combines raw authority, champion, economic buyer, and org alignment:

$$A_{blend} = 0.30 \cdot A + 0.30 \cdot Champ + 0.20 \cdot EB + 0.20 \cdot O$$

6. **Raw Score** — access term rescaled to $(0.5 + 0.5 \cdot Ac)$:

$$R_3 = A_{blend} \cdot (1 + N_{score}) \cdot (0.5 + 0.5 \cdot Ac)$$

7. **Log-Point** —

$$L_3 = \ln(1 + R_3)$$


In [74]:
def compute_p3_power(seniority: float, decision_involvement: float,
                     N: int, O: float, access: float,
                     champion_identified: int, champion_strength: float,
                     champion_active: float,
                     econ_buyer_identified: int, econ_buyer_engaged: float):

    A = seniority * decision_involvement
    Nscore = lognorm(N, 5)

    Champ = champion_identified * (0.50 * champion_strength
                                   + 0.50 * champion_active)

    EB = econ_buyer_identified * econ_buyer_engaged

    authority_blend = (0.30 * A
                       + 0.30 * Champ
                       + 0.20 * EB
                       + 0.20 * O)

    R3 = authority_blend * (1 + Nscore) * (0.5 + 0.5 * access)
    L3 = np.log(1 + R3)
    
    details = {
        "authority": A,
        "champion": Champ,
        "econ_buyer": EB,
        "Nscore": Nscore,
        "authority_blend": authority_blend,
        "R3_raw": R3,
        "L3_score": L3,
    }
    return L3


## Parameter 4: Momentum

**Question answered:** Is this deal moving forward, and how fast?

Merges Timeline/Urgency + Engagement because urgency without engagement is a ghost, and engagement without urgency is a tire-kicker.

## Variables

| Symbol | Description                         | Source        |
| ------ | ----------------------------------- | ------------- |
| $Td$   | Days until decision                 | Sales input   |
| $E$    | Engagement score (0–10 composite)   | CRM/Analytics |
| $V$    | Velocity (recent vs prior activity) | CRM           |
| $Ns$   | Next step defined? {0, 1}           | Sales input   |
| $Tr$   | Trigger event occurred? {0, 1}      | Sales input   |
| $DP$   | Decision process mapped \[0, 1]     | Sales input   |
| $PS$   | Procurement stage \[0, 1]           | Sales input   |
| $PP$   | Paper process clear \[0, 1]         | Sales input   |


## How It Works

1. **Timeline Pressure:** $T' = e^{-0.015 \cdot Td}$
2. **Engagement:** $E_{norm} = E/10$
3. **Velocity Bonus:** $V_{bonus} = \max(V - 1, 0)$
4. **Decision Process Composite:**

$$DProc = 0.40 \cdot DP + 0.35 \cdot PS + 0.25 \cdot PP$$

5. **Amplifiers** — $DProc$ added to the base sum:

$$R_4 = (T' + E_{norm} + V_{bonus} + DProc) \cdot (1 + 0.5 \cdot Ns + 0.5 \cdot Tr)$$

6. **Log-Point:** $L_4 = \ln(1 + R_4)$


In [75]:
def compute_p4_momentum(days_to_decision: float, engagement_score: float,
                        velocity: float, next_step: int, trigger: int,
                        decision_process_mapped: float,
                        procurement_stage: float,
                        paper_process_clear: float):

    Tprime = edp(days_to_decision, 0.015)
    Enorm = engagement_score / 10.0
    Vbonus = max(velocity - 1.0, 0.0)

    DProc = (0.40 * decision_process_mapped
             + 0.35 * procurement_stage
             + 0.25 * paper_process_clear)

    R4 = (Tprime + Enorm + Vbonus + DProc) * (1 + 0.5 * next_step
                                                + 0.5 * trigger)
    L4 = np.log(1 + R4)

    details = {
        "timeline_pressure": Tprime,
        "Enorm": Enorm,
        "Vbonus": Vbonus,
        "DProc": DProc,
        "R4_raw": R4,
        "L4_score": L4,
    }
    return L4


## Parameter 5: Edge (Weight = 0.10)

**Question answered:** Why would they choose us over alternatives?

Lower weight because startups often compete on fit and speed rather than brand.

### Variables

| Symbol | Description | Source |
|--------|-------------|--------|
| $$W$$ | Historical win rate $$[0, 1]$$ | CRM history |
| $$D$$ | Differentiation $$[0, 1]$$ | Sales/Product |
| $$Rel$$ | Relationship strength $$[0, 1]$$ | Sales input |
| $$Nc$$ | Number of competitors in the deal | Sales input |
| $$K$$ | Sole vendor flag {0, 1} | Sales input |

### How It Works

**If sole vendor** ($K = 1$):

$$R_5 = 2.0 \cdot (0.5 \cdot D + 0.5 \cdot Rel)$$

**If competitors exist** ($K = 0$):

1. Competitive pressure: $C_{press} = e^{-0.18 \cdot Nc}$
2. Win rate uses BCI when enough history exists, raw otherwise.
3. Raw score:

$$R_5 = 0.30 \cdot W_{adj} + 0.30 \cdot D + 0.20 \cdot Rel + 0.20 \cdot C_{press}$$

4. **Log-Point** — $L_5 = \ln(1 + R_5)$


In [76]:
def compute_p5_edge(win_rate: float, differentiation: float,
                    relationship: float, n_competitors: int,
                    sole_vendor: int, n_competitive_deals: int = None,
                    n_wins: int = None) -> tuple:
    if sole_vendor == 1:
        R5 = 2.0 * (0.5 * differentiation + 0.5 * relationship)
        details = {"mode": "sole_vendor", "R5_raw": R5}
    else:
        Cpress = edp(n_competitors, 0.18)
        if (n_competitive_deals is not None and n_wins is not None
                and n_competitive_deals > 5):
            Wadj = bci(n_wins, n_competitive_deals)
        else:
            Wadj = win_rate

        R5 = (0.30 * Wadj + 0.30 * differentiation
              + 0.20 * relationship + 0.20 * Cpress)

        details = {
            "mode": "competitive",
            "Cpress": Cpress,
            "Wadj": Wadj,
            "R5_raw": R5,
        }

    L5 = np.log(1 + R5)
    details["L5_score"] = L5
    return L5

## Gates and Modifier

### Gates (Binary Knockouts)

Reduced from 7 to 3 — only conditions that truly kill a deal at a startup:

| Gate | Condition that sets score to 0 |
|------|--------------------------------|
| G1 | No financial ability AND no funding path |
| G2 | Legal/compliance block — cannot legally sell to them |
| G3 | Technical infeasibility — product fundamentally cannot serve the need |

$$G = G_1 \times G_2 \times G_3$$

If any gate fails, the entire score becomes 0.

### Modifier (Deal Friction)

One combined deal friction score: $$M \in [0.5, 1.0]$$

| Value | Meaning |
|-------|---------|
| 1.0 | Clean, straightforward deal |
| 0.8 | Some friction (complex procurement) |
| 0.6 | Significant friction (multi-country, heavy legal) |
| 0.5 | Maximum friction before you'd gate it out |


## Complete Scoring Function

The final formula:

$$S = G \times M \times (0.25 \cdot L_1 + 0.20 \cdot L_2 + 0.25 \cdot L_3 + 0.20 \cdot L_4 + 0.10 \cdot L_5)$$


In [77]:
G = 1 if all(gates) else 0
M = max(0.5, min(1.0, modifier))

L1 = compute_p1_money(
    B=params["budget"],
    D=params["deal_value"],
    C_stated=params["budget_confirmation"],
    F=params["funding_security"],
    n_confirmed=params.get("n_confirmed_budget"),
    n_similar=params.get("n_similar_deals"),
)
L2 = compute_p2_fit(
    S=params["solution_fit"],
    P=params["problem_clarity"],
    n_feat_available=params["n_features_available"],
    n_feat_needed=params["n_features_needed"],
    D=params["dissatisfaction"],
    I=params["icp_match"],
    roi_quantified=params["roi_quantified"],
    metric_agreed=params["metric_agreed"],
    cost_of_inaction=params["cost_of_inaction"],
)
L3 = compute_p3_power(
    seniority=params["seniority"],
    decision_involvement=params["decision_involvement"],
    N=params["n_stakeholders"],
    O=params["org_alignment"],
    access=params["access_to_power"],
    champion_identified=params["champion_identified"],
    champion_strength=params["champion_strength"],
    champion_active=params["champion_active"],
    econ_buyer_identified=params["econ_buyer_identified"],
    econ_buyer_engaged=params["econ_buyer_engaged"],
)
L4 = compute_p4_momentum(
    days_to_decision=params["days_to_decision"],
    engagement_score=params["engagement_score"],
    velocity=params["velocity"],
    next_step=params["next_step_defined"],
    trigger=params["trigger_event"],
    decision_process_mapped=params["decision_process_mapped"],
    procurement_stage=params["procurement_stage"],
    paper_process_clear=params["paper_process_clear"],
)
L5 = compute_p5_edge(
    win_rate=params["win_rate"],
    differentiation=params["differentiation"],
    relationship=params["relationship_strength"],
    n_competitors=params["n_competitors"],
    sole_vendor=params["sole_vendor"],
    n_competitive_deals=params.get("n_competitive_deals"),
    n_wins=params.get("n_wins"),
)
weights = [0.25, 0.20, 0.25, 0.20, 0.10]
L_all = [L1, L2, L3, L4, L5]
labels = ["P1_Money", "P2_Fit", "P3_Power", "P4_Momentum", "P5_Edge"]
weighted_sum = sum(w * L for w, L in zip(weights, L_all))
score = 100 * G * M * weighted_sum
print(f"Final Score: {score:.2f}")


Final Score: 78.21
